# 02: Tonal-Aware Audio Processing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muchiriTimdev/ngumzo.ai/blob/main/notebooks/02_audio_processing.ipynb)

Process audio recordings to extract tonal features essential for tonal African languages like Kikongo, Yoruba, and Igbo.

## Learning Objectives
- Extract pitch contours for tone analysis
- Perform voice activity detection
- Segment audio by utterances
- Visualize tonal patterns
- Prepare data for speech-to-text training

## 1. Setup and Dependencies

In [ ]:
# Install dependencies
!pip install -q librosa soundfile numpy pandas matplotlib seaborn pyworld praat-parselmouth webrtcvad

# For Colab
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import signal
import librosa
import librosa.display
import soundfile as sf
import pyworld as pw
import parselmouth
from parselmouth.praat import call

# Set paths
DATA_ROOT = Path('/content/drive/MyDrive/african_oral_llm_data')
RAW_AUDIO_DIR = DATA_ROOT / 'raw_audio'
PROCESSED_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'

print("✓ Dependencies loaded")

## 2. Tonal Feature Extraction

For tonal languages, pitch (F0) is linguistically meaningful. Different tones can change word meaning:
- **Kikongo**: `kúnda` (to love) vs `kùnda` (to fall) - HIGH vs LOW tone
- **Yoruba**: `ọkọ` (husband) vs `ọ̀kọ̀` (hoe) - HIGH vs LOW tone

In [ ]:
class TonalFeatureExtractor:
    """
    Extracts pitch and tonal features from audio.
    Optimized for tonal African languages.
    """
    
    def __init__(self, sample_rate=48000):
        self.sample_rate = sample_rate
        self.frame_period = 5.0  # ms, for WORLD vocoder
    
    def extract_pitch_world(self, y, sr):
        """
        Extract pitch using WORLD vocoder (optimized for speech).
        
        Returns:
            f0: Fundamental frequency contour
            timeaxis: Time stamps for each frame
        """
        # Convert to float64 for WORLD
        y = y.astype(np.float64)
        
        # Harvest algorithm for F0 estimation
        f0, timeaxis = pw.harvest(y, sr, f0_floor=50, f0_ceil=600, 
                                   frame_period=self.frame_period)
        
        return f0, timeaxis
    
    def extract_pitch_praat(self, y, sr):
        """Extract pitch using Praat (linguistic gold standard)."""
        # Create Praat Sound object
        sound = parselmouth.Sound(y, sampling_frequency=sr)
        
        # To Pitch (cc) - cross-correlation method
        pitch = call(sound, "To Pitch", 0.0, 75, 600)
        
        # Get pitch values
        pitch_values = pitch.selected_array['frequency']
        time_stamps = pitch.xs()
        
        return pitch_values, time_stamps
    
    def extract_formants(self, y, sr):
        """Extract formant frequencies (vowel quality)."""
        sound = parselmouth.Sound(y, sampling_frequency=sr)
        
        # To Formant (burg)
        formant = call(sound, "To Formant (burg)", 0.0, 5, 5500, 0.025, 50)
        
        # Extract F1, F2, F3
        num_frames = call(formant, "Get number of frames")
        f1_values = []
        f2_values = []
        f3_values = []
        
        for i in range(1, num_frames + 1):
            t = call(formant, "Get time from frame number", i)
            f1 = call(formant, "Get value at time", 1, t, "Hertz", "Linear")
            f2 = call(formant, "Get value at time", 2, t, "Hertz", "Linear")
            f3 = call(formant, "Get value at time", 3, t, "Hertz", "Linear")
            
            f1_values.append(f1 if not np.isnan(f1) else 0)
            f2_values.append(f2 if not np.isnan(f2) else 0)
            f3_values.append(f3 if not np.isnan(f3) else 0)
        
        return np.array(f1_values), np.array(f2_values), np.array(f3_values)
    
    def extract_intensity(self, y, sr):
        """Extract intensity (loudness) contour."""
        sound = parselmouth.Sound(y, sampling_frequency=sr)
        intensity = call(sound, "To Intensity", 75, 0, True)
        
        intensity_values = intensity.values.T.flatten()
        time_stamps = intensity.xs()
        
        return intensity_values, time_stamps
    
    def analyze_tone_patterns(self, f0, timeaxis, segment_length=0.5):
        """
        Analyze tone patterns within segments.
        
        Returns dict with:
        - mean_pitch_per_segment
        - pitch_range_per_segment
        - tone_movement (rising/falling/level)
        """
        # Remove unvoiced frames
        voiced_f0 = f0[f0 > 0]
        
        if len(voiced_f0) == 0:
            return {
                'mean_pitch': 0,
                'pitch_range_semitones': 0,
                'tone_movement': 'unvoiced',
                'cv_pitch': 0  # Coefficient of variation
            }
        
        # Calculate pitch in semitones (relative pitch)
        pitch_semitones = 12 * np.log2(voiced_f0 / 100)  # Relative to 100Hz
        
        mean_pitch = np.mean(voiced_f0)
        pitch_range = np.max(pitch_semitones) - np.min(pitch_semitones)
        
        # Determine tone movement
        first_third = np.mean(pitch_semitones[:len(pitch_semitones)//3])
        last_third = np.mean(pitch_semitones[-len(pitch_semitones)//3:])
        
        pitch_diff = last_third - first_third
        if abs(pitch_diff) < 1:
            tone_movement = 'level'
        elif pitch_diff > 0:
            tone_movement = 'rising'
        else:
            tone_movement = 'falling'
        
        # Coefficient of variation (pitch stability)
        cv_pitch = np.std(voiced_f0) / mean_pitch if mean_pitch > 0 else 0
        
        return {
            'mean_pitch': float(mean_pitch),
            'pitch_range_semitones': float(pitch_range),
            'tone_movement': tone_movement,
            'cv_pitch': float(cv_pitch)
        }

extractor = TonalFeatureExtractor()
print("✓ TonalFeatureExtractor initialized")

## 3. Voice Activity Detection (VAD)

In [ ]:
try:
    import webrtcvad
    WEbrtc_AVAILABLE = True
except:
    WEbrtc_AVAILABLE = False
    print("webrtcvad not available, using energy-based VAD")

class VoiceActivityDetector:
    """
    Detect speech segments in audio.
    Essential for segmenting long recordings.
    """
    
    def __init__(self, aggressiveness=2):
        """
        Args:
            aggressiveness: 0-3, higher = more aggressive filtering
        """
        self.aggressiveness = aggressiveness
        if WEbrtc_AVAILABLE:
            self.vad = webrtcvad.Vad(aggressiveness)
    
    def energy_based_vad(self, y, sr, frame_duration=30, threshold=0.5):
        """Simple energy-based VAD (fallback method)."""
        frame_length = int(sr * frame_duration / 1000)
        
        # Calculate energy per frame
        energy = librosa.feature.rms(y=y, frame_length=frame_length, 
                                     hop_length=frame_length)[0]
        
        # Normalize
        energy_norm = (energy - energy.min()) / (energy.max() - energy.min() + 1e-8)
        
        # Binary voice decision
        is_voice = energy_norm > threshold
        
        # Convert to time segments
        times = np.arange(len(energy)) * frame_duration / 1000
        
        return is_voice, times, energy
    
    def get_speech_segments(self, y, sr, min_silence_duration=0.3, 
                           min_speech_duration=0.5):
        """
        Extract speech segments from audio.
        
        Returns list of (start_time, end_time) tuples.
        """
        is_voice, times, energy = self.energy_based_vad(y, sr)
        
        # Find segments
        segments = []
        in_speech = False
        segment_start = 0
        
        for i, voice in enumerate(is_voice):
            if voice and not in_speech:
                # Speech start
                segment_start = times[i]
                in_speech = True
            elif not voice and in_speech:
                # Speech end
                if times[i] - segment_start >= min_speech_duration:
                    segments.append((segment_start, times[i]))
                in_speech = False
        
        # Handle case where speech continues to end
        if in_speech and times[-1] - segment_start >= min_speech_duration:
            segments.append((segment_start, times[-1]))
        
        return segments, is_voice, times

vad = VoiceActivityDetector()
print("✓ Voice Activity Detector initialized")

## 4. Audio Visualization Tools

In [ ]:
def visualize_tonal_analysis(audio_path, segment_start=0, segment_duration=5):
    """
    Create comprehensive visualization of tonal features.
    
    Args:
        audio_path: Path to audio file
        segment_start: Start time in seconds
        segment_duration: Duration to analyze
    """
    # Load audio segment
    y, sr = librosa.load(audio_path, sr=48000, offset=segment_start, 
                        duration=segment_duration)
    
    # Extract features
    f0, timeaxis = extractor.extract_pitch_world(y, sr)
    intensity, intensity_times = extractor.extract_intensity(y, sr)
    
    # Get spectrogram
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    
    # Create figure
    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    
    # 1. Waveform
    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title('Waveform')
    axes[0].set_xlabel('')
    
    # 2. Spectrogram
    img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', 
                                    ax=axes[1], cmap='viridis')
    axes[1].set_title('Spectrogram')
    axes[1].set_ylim(0, 2000)  # Focus on speech frequencies
    fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
    
    # 3. Pitch contour
    # Only plot voiced frames
    voiced_indices = f0 > 0
    axes[2].plot(timeaxis[voiced_indices], f0[voiced_indices], 'b-', linewidth=1.5)
    axes[2].set_ylabel('F0 (Hz)')
    axes[2].set_title('Pitch Contour (Fundamental Frequency)')
    axes[2].set_ylim(50, 400)
    axes[2].grid(True, alpha=0.3)
    
    # 4. Intensity
    axes[3].plot(intensity_times, intensity, 'g-', linewidth=1.5)
    axes[3].set_ylabel('Intensity (dB)')
    axes[3].set_xlabel('Time (s)')
    axes[3].set_title('Intensity (Loudness)')
    axes[3].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print tonal analysis
    tone_analysis = extractor.analyze_tone_patterns(f0, timeaxis)
    print("\n🎵 Tonal Analysis:")
    for key, value in tone_analysis.items():
        print(f"  {key}: {value}")

# Example usage (when you have audio):
# visualize_tonal_analysis('/path/to/your/audio.wav')

## 5. Process Dataset

In [ ]:
class AudioPreprocessor:
    """
    Preprocess audio dataset with tonal feature extraction.
    """
    
    def __init__(self, output_dir):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.extractor = TonalFeatureExtractor()
        self.vad = VoiceActivityDetector()
    
    def process_recording(self, audio_path, metadata_path):
        """
        Process a single recording with all tonal features.
        
        Returns:
            processed_data: Dict with features and segment info
        """
        # Load metadata
        with open(metadata_path, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        
        # Load audio
        y, sr = librosa.load(audio_path, sr=48000)
        
        # Extract speech segments
        segments, _, _ = self.vad.get_speech_segments(y, sr)
        
        # Process each segment
        processed_segments = []
        
        for i, (start, end) in enumerate(segments):
            # Extract segment
            start_sample = int(start * sr)
            end_sample = int(end * sr)
            segment_audio = y[start_sample:end_sample]
            
            # Skip if too short
            if len(segment_audio) / sr < 0.5:
                continue
            
            # Extract features
            f0, timeaxis = self.extractor.extract_pitch_world(segment_audio, sr)
            tone_analysis = self.extractor.analyze_tone_patterns(f0, timeaxis)
            
            # Save segment audio
            segment_filename = f"{metadata['recording_id']}_seg{i:03d}.wav"
            segment_path = self.output_dir / 'segments' / segment_filename
            segment_path.parent.mkdir(parents=True, exist_ok=True)
            sf.write(segment_path, segment_audio, sr, subtype='PCM_24')
            
            processed_segments.append({
                'segment_id': f"{metadata['recording_id']}_seg{i:03d}",
                'start_time': float(start),
                'end_time': float(end),
                'duration': float(end - start),
                'audio_path': str(segment_path),
                'tone_analysis': tone_analysis,
                'f0_contour': f0.tolist(),  # Store full pitch contour
                'mean_f0': tone_analysis['mean_pitch'],
                'tone_movement': tone_analysis['tone_movement']
            })
        
        # Save processed info
        processed_data = {
            'recording_id': metadata['recording_id'],
            'original_path': str(audio_path),
            'num_segments': len(processed_segments),
            'total_speech_duration': sum(s['duration'] for s in processed_segments),
            'segments': processed_segments
        }
        
        # Save to JSON
        output_path = self.output_dir / f"{metadata['recording_id']}_processed.json"
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(processed_data, f, indent=2, ensure_ascii=False)
        
        return processed_data
    
    def process_dataset(self, metadata_dir, audio_dir):
        """Process all recordings in a dataset."""
        
        all_metadata = list(Path(metadata_dir).glob('*.json'))
        results = []
        
        for meta_path in all_metadata:
            try:
                with open(meta_path, 'r', encoding='utf-8') as f:
                    metadata = json.load(f)
                
                audio_path = Path(audio_dir) / f"{metadata['recording_id']}.wav"
                
                if not audio_path.exists():
                    print(f"⚠️ Audio not found: {audio_path}")
                    continue
                
                processed = self.process_recording(audio_path, meta_path)
                results.append(processed)
                print(f"✓ Processed: {metadata['title']['original']}")
                
            except Exception as e:
                print(f"❌ Error processing {meta_path}: {e}")
        
        return results

# Initialize preprocessor
preprocessor = AudioPreprocessor(PROCESSED_DIR)
print("✓ AudioPreprocessor initialized")

## 6. Batch Processing

In [ ]:
# Process entire dataset
# results = preprocessor.process_dataset(METADATA_DIR, RAW_AUDIO_DIR)

# Or process single recording for testing
# test_result = preprocessor.process_recording(
#     '/path/to/test.wav',
#     '/path/to/test_metadata.json'
# )

## 7. Export for Training

Prepare data in formats suitable for:
- Whisper fine-tuning
- Custom ASR training
- TTS training

In [ ]:
class DatasetExporter:
    """Export processed data for various training pipelines."""
    
    @staticmethod
    def export_for_whisper(processed_dir, transcripts_dir, output_dir):
        """
        Export in Whisper fine-tuning format.
        Requires transcripts to be completed first (Notebook 03).
        """
        
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        # Prepare data list
        training_data = []
        
        for processed_file in Path(processed_dir).glob('*_processed.json'):
            with open(processed_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            for segment in data['segments']:
                # Look for transcript
                transcript_path = Path(transcripts_dir) / f"{segment['segment_id']}.txt"
                
                if transcript_path.exists():
                    with open(transcript_path, 'r', encoding='utf-8') as f:
                        transcript = f.read().strip()
                    
                    training_data.append({
                        'audio': segment['audio_path'],
                        'text': transcript,
                        'tone_movement': segment.get('tone_movement', 'unknown')
                    })
        
        # Save as JSONL for Whisper
        output_file = output_dir / 'whisper_training_data.jsonl'
        with open(output_file, 'w', encoding='utf-8') as f:
            for item in training_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        print(f"✓ Exported {len(training_data)} samples to {output_file}")
        return training_data
    
    @staticmethod
    def create_manifest_csv(processed_dir, output_path):
        """Create CSV manifest for custom training."""
        
        records = []
        
        for processed_file in Path(processed_dir).glob('*_processed.json'):
            with open(processed_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            for segment in data['segments']:
                records.append({
                    'segment_id': segment['segment_id'],
                    'audio_path': segment['audio_path'],
                    'start_time': segment['start_time'],
                    'end_time': segment['end_time'],
                    'duration': segment['duration'],
                    'mean_f0': segment['mean_f0'],
                    'tone_movement': segment['tone_movement'],
                    'pitch_range_semitones': segment['tone_analysis']['pitch_range_semitones']
                })
        
        df = pd.DataFrame(records)
        df.to_csv(output_path, index=False)
        
        print(f"✓ Created manifest: {output_path}")
        print(f"  Total segments: {len(df)}")
        print(f"  Duration statistics:")
        print(df['duration'].describe())
        
        return df

exporter = DatasetExporter()
print("✓ DatasetExporter initialized")

## Summary

This notebook provides:
1. **Tonal feature extraction** using WORLD and Praat
2. **Voice activity detection** for speech segmentation
3. **Visualization tools** for pitch contours and tonal patterns
4. **Batch processing** for entire datasets
5. **Export utilities** for training pipelines

Next: Proceed to Notebook 03 for text preprocessing and transcription.